## GRPO Training — trading_rl (Jensen's alpha reward)

Trains a LoRA adapter with GRPO so the model writes Backtrader strategies with positive **Jensen's alpha** vs buy-and-hold of the same stock, then evaluates base vs GRPO on TEST (2022–25).

1. **GPU:** *Runtime → Change runtime type* → **A100** (or L4). vLLM generation needs compute capability ≥ 8, so a T4 is only good for checking the plumbing.
2. **Secrets:** `hf_token`, `alpaca_key`, `alpaca_secret` (Colab: key icon; Kaggle: *Add-ons → Secrets*, and turn *Internet* on).
3. **Settings** cell: `MODEL`, `NUM_STEPS`, `HUB_REPO`. Do a first run with `NUM_STEPS = 2` to check everything loads, then restart the runtime and run the full budget.
4. Run every cell top to bottom. Results are zipped and downloaded at the end.

> `trading_rl` is installed from GitHub, so local changes must be pushed first.

In [ ]:
# vllm-bnb-plugin: vLLM >= 0.28 needs it for 4-bit (bitsandbytes) models with fast_inference
# Install trading_rl (+ [train] extra: unsloth, trl, datasets, peft) + vLLM (vLLM makes GRPO generation on CUDA several times faster)
%pip install -q "trading-rl[train] @ git+https://github.com/adhamhelmy/llm-fine-tuning.git" unsloth "trl==0.22.2" datasets peft vllm vllm-bnb-plugin

In [ ]:
# Shared backtest / prompt / reward code: the `trading_rl` package in this repo
import os, sys
if os.path.isdir('../trading_rl'):   # local clone: import straight from the repo
    sys.path.insert(0, os.path.abspath('..'))
else:                                # Colab / Kaggle
    %pip install -q "trading-rl @ git+https://github.com/adhamhelmy/llm-fine-tuning.git"

### Secrets

Colab *Secrets* or Kaggle *Add-ons → Secrets* hold these; the cell below fetches them into `os.environ`. They never touch the notebook.

In [ ]:
import os
try:                                   # Colab: Secrets (key icon)
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:                    # Kaggle: Add-ons -> Secrets (attach them to this notebook)
    from kaggle_secrets import UserSecretsClient
    get_secret = UserSecretsClient().get_secret

# Colab secret names are case-sensitive: accept either the lowercase name or the env-var name
SECRETS = {'HF_TOKEN': 'hf_token', 'ALPACA_API_KEY': 'alpaca_key', 'ALPACA_SECRET_KEY': 'alpaca_secret'}
for env, alias in SECRETS.items():
    if os.environ.get(env):
        continue
    for name in (env, alias):
        try:
            os.environ[env] = get_secret(name)
            break
        except Exception as e:
            err = f'{type(e).__name__}: {e}'
    else:
        print(f'secret {env!r} (or {alias!r}) not readable -> {err}')

needed = {k: os.environ.get(k) for k in ('HF_TOKEN', 'ALPACA_API_KEY', 'ALPACA_SECRET_KEY')}
missing = [k for k, v in needed.items() if not v]
print('all secrets present' if not missing else 'MISSING: ' + ', '.join(missing))

### Settings

One GRPO step = 1 prompt × `num_generations` (8) completions. Rough A100 timing for a 3B model: ~1 min/step (≈3.5 h for 200 steps); a 7B model is ~2× slower. With `HUB_REPO` set, a checkpoint is pushed every `SAVE_STEPS`, so an interrupted run can resume.

In [ ]:
import torch
from trading_rl.model import Unsloth        # BEFORE trl / transformers
from trading_rl import (TEST_END, TEST_START, TEST_SYMBOLS, TRAIN_END, TRAIN_START, TRAIN_UNIVERSE,
                        Backtester, LadderConfig, RewardConfig, make_strategy_reward, prompt_fn)
from trading_rl.dataset import build_dataset, train_windows

MODEL       = 'unsloth/Ministral-3-3B-Instruct-2512'
NUM_STEPS   = 200      # 2 = plumbing check; 150-250 for the real run
SAVE_STEPS  = 25       # checkpoint (and Hub push, if HUB_REPO is set) every N steps
HUB_REPO    = ''       # e.g. 'your-hf-username/ministral3-3b-grpo-alpha' ('' = don't push)
RESUME      = False    # True: continue from the last checkpoint in outputs/ (or pulled from HUB_REPO)
EVAL_ONLY   = False    # True: skip training, load the trained adapter from HUB_REPO, then evaluate
# vLLM generation (several times faster) needs compute capability >= 8 (L4/A100/H100).
FAST_INFERENCE = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print('CUDA?', torch.cuda.is_available(), '| vLLM:', FAST_INFERENCE)

# Reward for training AND evaluation: Jensen's alpha (annualised %, vs buy-and-hold of the
# same symbol) for every strategy that trades; negative alphas squashed into (-0.99, 0) so
# trading still beats not trading (-1); capped at 50 %/yr. 'low' costs (1 + 2 bps per side).
REWARD = RewardConfig.alpha()

# GRPO: 8 completions per prompt (G=2 turned every reward gap into +-0.707), rewards scaled
# over the batch so the size of an alpha gap counts.
GRPO = dict(num_generations=8, per_device_train_batch_size=1, gradient_accumulation_steps=8,
            scale_rewards='batch', learning_rate=5e-6)

# Prompt + context for training (same as evaluation below). LadderConfig() = bare prompt.
PROMPT_CONFIG = LadderConfig(api_reference=True)

# Training data: ~340 large/mid-cap US stocks (TRAIN_UNIVERSE), none of them a TEST stock, on
# 2016-21 and each half (2016-18, 2019-21). Symbols with missing / late / forward-filled bars over
# the full window are dropped; the halves are cut from the full window (no extra downloads).
# Downloads ~340 symbols from Alpaca on first run (a few minutes).
backtester = Backtester(verbose=False)
WINDOWS = train_windows(TRAIN_START, TRAIN_END)
TRAIN_SYMBOLS, train_dropped = backtester.clean_symbols(TRAIN_UNIVERSE, *WINDOWS[0])
assert len(TRAIN_SYMBOLS) > 200, f'only {len(TRAIN_SYMBOLS)} usable training symbols: Alpaca keys / network?'
assert not set(TRAIN_SYMBOLS) & set(TEST_SYMBOLS)
dataset = build_dataset(TRAIN_SYMBOLS, prompt_fn=prompt_fn(PROMPT_CONFIG, backtester),
                        windows=WINDOWS, shuffle_seed=0)
print(len(TRAIN_SYMBOLS), 'training symbols x', len(WINDOWS), 'windows =', len(dataset), 'prompts;',
      NUM_STEPS, 'steps use', min(NUM_STEPS, len(dataset)), 'of them')

### Load the model

In [ ]:
ADAPTER = None
if EVAL_ONLY:   # new session: fetch the trained adapter (pushed at the end of training) into grpo_saved_lora/
    from huggingface_hub import snapshot_download
    assert HUB_REPO, 'EVAL_ONLY needs HUB_REPO (where training pushed the adapter)'
    ADAPTER = snapshot_download(HUB_REPO, local_dir='grpo_saved_lora', token=os.environ.get('HF_TOKEN'),
                                ignore_patterns=['last-checkpoint/*', 'checkpoint-*/*'])
model = Unsloth(model_name=MODEL, max_seq_length=5120, max_prompt_length=3072,
                fast_inference=FAST_INFERENCE, gpu_memory_utilization=0.6,
                lora_adapter_path=ADAPTER, adapter_trainable=False, hf_token=os.environ.get('HF_TOKEN'))

### Train

Saves the LoRA adapter to `grpo_saved_lora/` (and pushes it to `HUB_REPO`). Profitable strategies found during training go to `successful_strategies/`.

In [ ]:
import os
push = dict(hub_model_id=HUB_REPO, hf_token=os.environ.get('HF_TOKEN'), hub_strategy='checkpoint') if HUB_REPO else {}
if EVAL_ONLY:
    print('EVAL_ONLY: skipping training')
else:
  trainer = model.train(
    steps=NUM_STEPS,
    reward_functions=[make_strategy_reward(backtester, reward=REWARD, save_dir='successful_strategies',
                                           save_plots=False)],
    dataset=dataset,
    save_steps=SAVE_STEPS,
    resume_from_checkpoint=RESUME,
    **push,
    **GRPO,
)

### Evaluate: base vs GRPO on TEST

Same prompt, reward and token budget as training; the base model is the same weights with the adapter switched off.

- **Universe:** `TEST_SYMBOLS` = S&P 100 + Dow 30 as of January 2022 (101 tickers, point-in-time). None of them is a training stock. Symbols whose 2022–25 bars are missing or forward-filled are dropped and listed in `symbols_test.csv`.
- **Window:** TEST 2022–25, never used for any choice. (The old DEV years 2020–21 are now training data.)
- **Samples:** `N_SAMPLES` strategies per (model, prompt, symbol). Statistics are paired **by symbol** (bootstrap CI, Wilcoxon), so the stock count is the sample size.
- **Memorization probe:** every (model, symbol, sample) also runs with the ticker and dates removed (`anonymize=True`). If alpha holds up without knowing which stock and when, results don't rest on remembered prices.
- Two pre-registered comparisons (base vs GRPO; named vs anonymous), no search over configs: no multiple-testing deflation needed.

In [ ]:
import os
import pandas as pd
from trading_rl import (ScoringPool, baseline_table, format_report, paired_delta, report,
                        run_ablation_batched)
from trading_rl.robustness import paired_wilcoxon, signed_rank_test

ANON_CONFIG  = PROMPT_CONFIG.replace(anonymize=True)   # memorization probe: no ticker, no dates
EVAL_CONFIGS = [PROMPT_CONFIG, ANON_CONFIG]
N_SAMPLES    = 8
EVAL_TOKENS  = model.max_seq_length - model.max_prompt_length - 1   # = training completion budget
LABEL        = MODEL.split('/')[-1]
OUT          = f'ladder_results/{LABEL}'
os.makedirs(OUT, exist_ok=True)


def show(df, baselines, title):
    print(f'\n=== {title}: {df["symbol"].nunique()} symbols ===')
    print(format_report(report(df, baselines, by='model'), by='model').T.to_string())
    for metric in ('profitable', 'beat_bh', 'alpha_pct', 'reward_score'):
        print(' ', paired_delta(df, 'base', 'grpo', metric=metric, baselines=baselines, by='model'))
    print(' ', paired_wilcoxon(df, 'base', 'grpo', metric='alpha_pct', by='model'))
    for m in ('base', 'grpo'):   # is mean per-symbol alpha of traded strategies > 0?
        per = df[(df['model'] == m) & df['status'].isin(['profitable', 'loss'])].groupby('symbol')['alpha_pct'].mean()
        print(f'  {m}: per-symbol alpha vs 0: median {per.median():.2f} %/yr, n={len(per)}, '
              f'Wilcoxon p={signed_rank_test(per)["p"]:.3g}')


tables = {}
for split, (start, end) in {'test': (TEST_START, TEST_END)}.items():
    symbols, dropped = backtester.clean_symbols(TEST_SYMBOLS, start, end)
    pd.DataFrame([(s, 'used', '') for s in symbols] + [(s, 'dropped', why) for s, why in dropped],
                 columns=['symbol', 'status', 'issue']).to_csv(f'{OUT}/symbols_{split}.csv', index=False)
    baselines = baseline_table(backtester, symbols, start, end, reward=REWARD)
    baselines.to_csv(f'{OUT}/baselines_{split}.csv', index=False)
    csv = f'{OUT}/eval_{split}.csv'
    with ScoringPool(workers=max(1, (os.cpu_count() or 2) - 1)) as pool:
        for label, adapter in (('base', False), ('grpo', True)):
            print(f'\n### {split}: {label}')
            run_ablation_batched(
                lambda prompts: model.generate_batch(prompts, temperature=1.0, max_new_tokens=EVAL_TOKENS,
                                                     adapter=adapter),
                backtester, EVAL_CONFIGS, symbols, start, end, N_SAMPLES, csv,
                model=label, reward=REWARD, pool=pool)
    df = pd.read_csv(csv)
    df = df[(df['reward'] == REWARD.name) & (df['end'] == end)]
    tables[split] = (df, baselines)
    for cfg in EVAL_CONFIGS:
        d = df[df['config'] == cfg.name]
        tag = 'anonymous' if cfg.anonymize else 'named'
        show(d, baselines, f'{split.upper()} {start} -> {end}, {tag} prompt')
    # memorization probe: does alpha drop when the model doesn't know which stock / which years?
    print(f'\n=== {split.upper()} memorization probe: anonymous - named (paired by symbol) ===')
    for m in ('base', 'grpo'):
        for metric in ('alpha_pct', 'profitable', 'reward_score'):
            print(f'  {m}:', paired_delta(df[df['model'] == m], PROMPT_CONFIG.name, ANON_CONFIG.name,
                                          metric=metric, baselines=baselines, by='config'))

### Robustness report + download

Re-backtests every strategy and buy-and-hold at all cost levels (alpha, beta, Wilcoxon vs B&H, by-year, near-duplicates), then zips `ladder_results/`, `successful_strategies/` and the adapter.

In [ ]:
import shutil
from trading_rl.robustness import full_report

for split in tables:
    full_report(f'{OUT}/eval_{split}.csv', backtester, out_dir=f'{OUT}/robustness_{split}')
    print(open(f'{OUT}/robustness_{split}/report.md').read()[:3000])

shutil.make_archive('grpo_results', 'zip', '.', 'ladder_results')
for extra in ('successful_strategies', 'grpo_saved_lora'):
    if os.path.isdir(extra):
        shutil.make_archive(extra, 'zip', '.', extra)
try:
    from google.colab import files
    for f in ('grpo_results.zip', 'successful_strategies.zip', 'grpo_saved_lora.zip'):
        if os.path.exists(f):
            files.download(f)
except ImportError:
    print('zips written to', os.getcwd())